<a href="https://colab.research.google.com/github/kokila83/AgenticAIMani_SampleMiniProject/blob/main/Ex%206_Production%20Agent%20Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import getpass
import json
from openai import OpenAI

# 1. Get Azure AI Foundry API key securely
api_key = getpass.getpass("Enter your Azure AI Foundry API key: ")

# 2. Configure OpenAI client pointing to your Azure AI Foundry endpoint
client = OpenAI(
    api_key=api_key.strip(),
    base_url="https://ganapathy-0333-resource.services.ai.azure.com/openai/v1/"
)

# 3. Production Agent Class
class ProductionAgent:
    def __init__(self, name, system_prompt):
        self.name = name
        self.client = client
        self.system_prompt = system_prompt
        self.tools = {}

    def register_tool(self, name, func, description):
        self.tools[name] = {"func": func, "description": description}

    def run(self, task):
        tools_desc = '\n'.join([f"- {n}: {t['description']}" for n, t in self.tools.items()])

        prompt = f"""{self.system_prompt}

Tools:
{tools_desc}

Task: {task}

Respond ONLY with valid JSON in this format: {{"thought": "reasoning", "action": "tool_name", "input": "value"}}"""

        # Call Azure AI Foundry model
        response = self.client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[{"role": "user", "content": prompt}]
        )

        try:
            raw_content = response.choices[0].message.content.strip()
            decision = json.loads(raw_content)
            print(f"💭 Thought: {decision['thought']}")

            if decision['action'] in self.tools:
                result = self.tools[decision['action']]['func'](decision['input'])
                print(f"🔧 Tool {decision['action']}: {result}")
                return result
            else:
                print(f"✅ Final: {decision['input']}")
                return decision['input']
        except Exception as e:
            return f"Error parsing response: {e}\nRaw output: {response.choices[0].message.content}"

# 4. Instantiate and run the agent
if __name__ == "__main__":
    agent = ProductionAgent("Assistant", "You are a helpful assistant.")
    agent.register_tool("calculator", lambda x: str(eval(x)), "Evaluates math expressions")
    agent.run("Calculate 15% tip on $85 bill")

Enter your Azure AI Foundry API key: ··········
💭 Thought: Calculate 15% of 85 to find the tip amount.
🔧 Tool calculator: 12.75
